<a href="https://colab.research.google.com/github/Prezzo-K/Graph-Neural-Networks-for-Fraud-Detection/blob/classical-ml-experiments/notebooks/Traditional_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# import scikit learn and check version
import sklearn as sk

sk.__version__

'1.6.1'

In [8]:
# Workflow
# 1. Get the dataset, do preprocessing if any and split it into train and test set
# 2. Load various clasical ML models - Random Forest (RF), Extra Trees, Logistic Regression (LR), Support Vector Machine (SVM), XGBoost
# 3. Train and test them on the test set
# 3. Compare their Accuracy, Precision and Recall, F1-Score, Area Under the ROC Curve (AUC–ROC), Confusion Matrix


# Load Dataset

In [9]:
import os
import pandas as pd

# Dynamically find the dataset path relative to the notebook's directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
file_path = os.path.join(project_root, "data", "Fraud Detection Transactions Dataset.csv")

dataset  = pd.read_csv(filepath_or_buffer = "/content/data/Fraud Detection Transactions Dataset.csv")
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Transaction_ID                50000 non-null  object 
 1   User_ID                       50000 non-null  object 
 2   Transaction_Amount            50000 non-null  float64
 3   Transaction_Type              50000 non-null  object 
 4   Timestamp                     50000 non-null  object 
 5   Account_Balance               50000 non-null  float64
 6   Device_Type                   50000 non-null  object 
 7   Location                      50000 non-null  object 
 8   Merchant_Category             50000 non-null  object 
 9   IP_Address_Flag               50000 non-null  int64  
 10  Previous_Fraudulent_Activity  50000 non-null  int64  
 11  Daily_Transaction_Count       50000 non-null  int64  
 12  Avg_Transaction_Amount_7d     50000 non-null  float64
 13  F

In [10]:
# Retrieve the feature and labels separate

X = dataset.loc[:,"Transaction_ID":"Is_Weekend"]
y = dataset.loc[:, "Fraud_Label"]

X.head(2)

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,Account_Balance,Device_Type,Location,Merchant_Category,IP_Address_Flag,Previous_Fraudulent_Activity,Daily_Transaction_Count,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend
0,TXN_33553,USER_1834,39.79,POS,2023-08-14 19:30:00,93213.17,Laptop,Sydney,Travel,0,0,7,437.63,3,Amex,65,883.17,Biometric,0.8494,0
1,TXN_9427,USER_7875,1.19,Bank Transfer,2023-06-07 04:01:00,75725.25,Mobile,New York,Clothing,0,0,13,478.76,4,Mastercard,186,2203.36,Password,0.0959,0


In [11]:
y.head(2)

,Fraud_Label
0,0
1,1


In [12]:
y.value_counts()

,count
Fraud_Label,
0,33933
1,16067


In [13]:
# Standardize columns with continous values - Transaction_Amount, Account_Balance, Daily_Transaction_Count, Avg_Transaction_Amount_7d, Card_Age, Transaction_Distance, Risk_Score
# Encode these as one hot encoding - Transaction_Type, Device_Type, Location, Merchant_Category, Card_Type, Authentication_Method
# Leave Binary / Numeric columns as they are. Don't scale or encode - IP_Address_Flag, Previous_Fraudulent_Activity, - Is_Weekend

# Drop the Transaction_ID, User_ID as they don't add in classification
X.drop(columns = ["Transaction_ID", "User_ID"], inplace = True)

# Remove risk score feature to avoid data leakge
# this feature seems to do data leakage though it will be beneficial in real world scenarios
# We also remove Failed_Transaction_Count_7d because it perfectly predicts fraud and dominates the models
X.drop(columns = ["Risk_Score", "Failed_Transaction_Count_7d"], inplace = True)

# Convert 'Timestamp' to datetime and extract various fields
X["Timestamp"] = pd.to_datetime(X["Timestamp"])
X["Hour"] = X["Timestamp"].dt.hour
X["Day"] = X["Timestamp"].dt.day
X["Month"] = X["Timestamp"].dt.month
X["DayOfWeek"] = X["Timestamp"].dt.dayofweek
# drop off timestamp col now
X.drop(columns = ["Timestamp"], inplace = True)

# Do one hot encoding
categorical_cols = [
    "Transaction_Type",
    "Device_Type",
    "Location",
    "Merchant_Category",
    "Card_Type",
    "Authentication_Method"
]
X = pd.get_dummies(X, columns = categorical_cols)
# Display the first few rows with the new 'Hour' column to confirm
X

# Scale the dataset
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)


In [14]:
# split the data into train and test set
from sklearn.model_selection import train_test_split

random_state = 42
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = test_size,
                                                    shuffle = True,
                                                    stratify = y
                                                    )
len(X_train), len(y_train), len(X_test), len(y_test)

(40000, 40000, 10000, 10000)

In [15]:
# do some sanity check to ensure good splits of the classes in train and test sets
y_train.value_counts(), y_test.value_counts()

(Fraud_Label
 0    27146
 1    12854
 Name: count, dtype: int64,
 Fraud_Label
 0    6787
 1    3213
 Name: count, dtype: int64)

# Load Models And Train

## Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier

# class_weight='balanced' scales each class inversely by its frequency:
#   weight = n_samples / (n_classes * n_samples_per_class)
# Forces the model to pay more attention to the minority (fraud) class
rf = RandomForestClassifier(random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [17]:
preds_rf = rf.predict(X_test)

In [18]:
# metrics -  Accuracy, Precision and Recall, F1-Score, Area Under the ROC Curve (AUC–ROC), Confusion Matrix
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

# List to hold results
model_results = []

def evaluate_model(y_true, preds, model_name="Model"):
    accuracy = accuracy_score(y_true, preds)
    f1 = f1_score(y_true, preds, zero_division=0)
    precision = precision_score(y_true, preds, zero_division=0)
    recall = recall_score(y_true, preds, zero_division=0)
    report = classification_report(y_true, preds, zero_division=0)

    print(f"\n{model_name} Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("\nClassification Report:\n", report)
    
    # Store the results
    result_dict = {
        "Model": model_name,
        "Accuracy": accuracy,
        "F1-Score": f1,
        "Precision": precision,
        "Recall": recall
    }
    model_results.append(result_dict)
    
    return result_dict

# Random Forest Evaluation (which was created right before this cell typically)
evaluate_model(y_test, preds_rf, model_name="Random Forest Classifier")



Random Forest Classifier Metrics:
Accuracy: 0.6785
F1-Score: 0.0006
Precision: 0.2500
Recall: 0.0003

Classification Report:
               precision    recall  f1-score   support

           0       0.68      1.00      0.81      6787
           1       0.25      0.00      0.00      3213

    accuracy                           0.68     10000
   macro avg       0.46      0.50      0.40     10000
weighted avg       0.54      0.68      0.55     10000



{'Model': 'Random Forest Classifier',
 'Accuracy': 0.6785,
 'F1-Score': 0.0006216972334473111,
 'Precision': 0.25,
 'Recall': 0.0003112356053532524}

## Extra Trees

In [19]:
from sklearn.ensemble import ExtraTreesClassifier

ext_clf = ExtraTreesClassifier(random_state=42, class_weight='balanced')
ext_clf.fit(X_train, y_train)

preds_ext_clf = ext_clf.predict(X_test)
evaluate_model(y_test, preds_ext_clf, model_name='Extra Trees Classifier')


Extra Trees Classifier Metrics:
Accuracy: 0.6743
F1-Score: 0.0234
Precision: 0.3197
Recall: 0.0121

Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.99      0.80      6787
           1       0.32      0.01      0.02      3213

    accuracy                           0.67     10000
   macro avg       0.50      0.50      0.41     10000
weighted avg       0.56      0.67      0.55     10000



{'Model': 'Extra Trees Classifier',
 'Accuracy': 0.6743,
 'F1-Score': 0.023388305847076463,
 'Precision': 0.319672131147541,
 'Recall': 0.012138188608776844}

## XGBoost

In [20]:
!pip install -q xgboost

In [21]:
from xgboost import XGBClassifier

# XGBoost doesn't use class_weight — it uses scale_pos_weight instead.
# Formula: count(negatives) / count(positives)
# Same concept as class_weight='balanced' but XGBoost's own implementation
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'Class ratio — Legit: {neg}, Fraud: {pos}, scale_pos_weight: {scale_pos_weight:.2f}')

xg_bst = XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=scale_pos_weight)
xg_bst.fit(X_train, y_train)

xg_bst_preds = xg_bst.predict(X_test)
evaluate_model(y_test, xg_bst_preds, model_name='XGBoost Classifier')

Class ratio — Legit: 27146, Fraud: 12854, scale_pos_weight: 2.11

XGBoost Classifier Metrics:
Accuracy: 0.5468
F1-Score: 0.3604
Precision: 0.3297
Recall: 0.3974

Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.62      0.65      6787
           1       0.33      0.40      0.36      3213

    accuracy                           0.55     10000
   macro avg       0.51      0.51      0.50     10000
weighted avg       0.57      0.55      0.56     10000



{'Model': 'XGBoost Classifier',
 'Accuracy': 0.5468,
 'F1-Score': 0.36042901495907426,
 'Precision': 0.32971856442034597,
 'Recall': 0.3974478680361033}

## Logistic Regression

In [22]:
from sklearn.linear_model import LogisticRegression

# max_iter=1000 because the solver needs more iterations to converge on this feature set
lr = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
lr.fit(X_train, y_train)

preds_lr = lr.predict(X_test)
evaluate_model(y_test, preds_lr, model_name='Logistic Regression')


Logistic Regression Metrics:
Accuracy: 0.5030
F1-Score: 0.3952
Precision: 0.3245
Recall: 0.5054

Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.50      0.58      6787
           1       0.32      0.51      0.40      3213

    accuracy                           0.50     10000
   macro avg       0.50      0.50      0.49     10000
weighted avg       0.57      0.50      0.52     10000



{'Model': 'Logistic Regression',
 'Accuracy': 0.503,
 'F1-Score': 0.39522998296422485,
 'Precision': 0.32447552447552447,
 'Recall': 0.5054466230936819}

## Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import SVC

# probability=True needed if you want predict_proba() for AUC-ROC later
svm_clf = SVC(random_state=42, class_weight='balanced', probability=True)
svm_clf.fit(X_train, y_train)

preds_svm = svm_clf.predict(X_test)
evaluate_model(y_test, preds_svm, model_name='SVM Classifier')

In [ ]:
import os

# Create the results directory if it doesn't exist
results_dir = os.path.join(os.path.dirname(os.getcwd()), "results")
os.makedirs(results_dir, exist_ok=True)

results_file_path = os.path.join(results_dir, "traditional_ml_results.txt")

# Write results to the file
with open(results_file_path, "w") as f:
    f.write("Traditional ML Models Performance\n")
    f.write("=================================\n\n")
    for res in model_results:
        f.write(f"Model: {res['Model']}\n")
        f.write(f"  Accuracy:  {res['Accuracy']:.4f}\n")
        f.write(f"  F1-Score:  {res['F1-Score']:.4f}\n")
        f.write(f"  Precision: {res['Precision']:.4f}\n")
        f.write(f"  Recall:    {res['Recall']:.4f}\n")
        f.write("-" * 35 + "\n")

print(f"Results successfully saved to {results_file_path}")


In [ ]:
import joblib

# Create a 'models' directory at the root of the project
models_dir = os.path.join(project_root, "models")
os.makedirs(models_dir, exist_ok=True)

# Dictionary of all our trained models
trained_models = {
    "random_forest": rf,
    "extra_trees": ext_clf,
    "xgboost": xg_bst,
    "logistic_regression": lr,
    "svm": svm_clf
}

# Save each model using joblib
for model_name, model_obj in trained_models.items():
    model_path = os.path.join(models_dir, f"{model_name}.joblib")
    joblib.dump(model_obj, model_path)
    print(f"Successfully saved {model_name} to {model_path}")
